# Phase 2: Stylometric Feature Engineering & Statistical Analysis (RQ 1.2)

This notebook extracts **34 interpretable stylometric and structural features** organized into **5 domain-grounded feature groups** across our balanced corpus ($N = 6,000$ tracks: $3,000$ TikTok-Viral vs. $3,000$ Non-Viral commercial tracks), and conducts our internal statistical validation:
1. **95% Confidence Intervals** ($\bar{x} \pm 1.96 \cdot \frac{s}{\sqrt{n}}$) & Cohen's $d$ effect sizes across all 5 feature groups.
2. **Chi-Square ($\chi^2$) Test of Independence** ($2 \times 2$ contingency tables) to identify the most statistically significant lexical markers of TikTok virality.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.build_features import (
    ALL_FEATURE_COLUMNS,
    FEATURE_GROUPS,
    run_feature_pipeline,
)
from src.features.statistical_tests import (
    compute_chi_square_vocabulary,
    compute_feature_confidence_intervals,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
print(f"Loaded Phase 2 pipeline ({len(ALL_FEATURE_COLUMNS)} features across {len(FEATURE_GROUPS)} groups).")

## 1. Extract All 5 Interpretable Feature Groups
Runs lexical, repetition/zlib compression, pronoun/function word, `spaCy` POS, and CMU phonetic rhyme extractors on `dataset_full.parquet` and saves `features_full.parquet`, `features_train.parquet`, and `features_test.parquet`.

In [ ]:
splits = run_feature_pipeline(force_recompute=False)
features_full = splits["full"]
features_train = splits["train"]
features_test = splits["test"]

features_full[["track_name", "primary_artist", "label_name"] + ALL_FEATURE_COLUMNS[:8]].head()

## 2. 95% Confidence Intervals & Effect Sizes Across Feature Groups (RQ 1.2)
Computes group means, 95% Confidence Intervals ($\bar{x} \pm 1.96 \cdot \text{SE}$), Cohen's $d$, and Welch's $t$-test $p$-values comparing Viral (`label=1`) vs. Non-Viral (`label=0`).

In [ ]:
ci_df = compute_feature_confidence_intervals(
    df=features_full,
    feature_groups=FEATURE_GROUPS,
    label_col="label",
)

# Save summary table for LaTeX paper reporting
ci_out_path = PROJECT_ROOT / "data" / "processed" / "stat_ci_summary.csv"
ci_df.to_csv(ci_out_path, index=False)
print(f"Saved 95% CI summary table to {ci_out_path}")

display_cols = [
    "feature_group",
    "feature",
    "viral_mean",
    "non_viral_mean",
    "diff_mean",
    "diff_ci95_low",
    "diff_ci95_high",
    "cohens_d",
    "p_value",
    "ci_non_overlapping",
]
ci_df[display_cols].head(20)

## 3. Visualizing Standardized Effect Sizes (Cohen's $d$ with 95% CI)

In [ ]:
top_feats = ci_df.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d95f02" if d > 0 else "#1b9e77" for d in top_feats["cohens_d"]]
ax.barh(top_feats["feature"], top_feats["cohens_d"], color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Cohen's d (Positive = Higher in TikTok Viral, Negative = Higher in Non-Viral)")
ax.set_title("Top 15 Stylometric Features Distinguishing TikTok-Viral from Non-Viral Lyrics")
plt.tight_layout()
plt.show()

## 4. Chi-Square ($\chi^2$) Test of Independence on Vocabulary
Identifies the single words whose document frequency differs most significantly between Viral (`label=1`) and Non-Viral (`label=0`) tracks using a $2 \times 2$ contingency table.

In [ ]:
chi2_df = compute_chi_square_vocabulary(
    df=features_full,
    text_col="lyrics",
    label_col="label",
    min_doc_freq=30,
    top_k=100,
)

chi2_out_path = PROJECT_ROOT / "data" / "processed" / "stat_chi2_vocab.csv"
chi2_df.to_csv(chi2_out_path, index=False)

print("--- Top 15 Words Associated with TikTok-Viral Songs (label=1) ---")
display(chi2_df[chi2_df["direction"] == "Viral (label=1)"].head(15))

print("\n--- Top 15 Words Associated with Non-Viral Commercial Songs (label=0) ---")
display(chi2_df[chi2_df["direction"] == "Non-Viral (label=0)"].head(15))